# Crop suitability trends and machine learning (Sri Lanka)
**What this notebook does**
1. Loads the crop requirement table and 35 years of NASA POWER climate for your location.
2. Tests whether the climate is *actually* trending (Theil–Sen slope + Mann–Kendall test) and extends the trend to a target year with an uncertainty band.
3. Shows how often each crop's ranges were met in past seasons, and whether that frequency is rising or falling.
4. Uses machine learning where the data supports it: **unsupervised** clustering/PCA of crop requirements, and a **back-tested** rainfall forecast compared with a simple baseline.

**Honest limits.** The CSV has no yields or outcomes, so no supervised crop-yield model is possible. Most crop ranges are estimates, and 40 crops is a small sample. Trend extensions are statistical extrapolations, not climate projections. Use results for screening and check with local agricultural officers.

## 1. Settings and load the crop CSV

In [ ]:
import io, re, warnings
import numpy as np, pandas as pd, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
warnings.filterwarnings("ignore")

# ---------------- EDIT THESE ----------------
LAT, LON      = 7.29, 80.63          # your field location (example: Kandy area)
START, END    = 1991, 2025           # full calendar years of NASA POWER data
SEASON_MONTHS = [10, 11, 12, 1, 2, 3]  # Maha ~Oct-Mar; Yala: [4,5,6,7,8,9]
PH_BAND       = (5.5, 6.5)           # from YOUR soil test (NASA cannot provide pH)
TARGET_YEAR   = 2032                 # how far to extend trends
RECENT_N      = 10                   # "recent" = last N seasons
# --------------------------------------------

try:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(io.BytesIO(next(iter(up.values()))))
except ImportError:
    df = pd.read_csv("sri_lanka_crop_cultivation_requirements.csv")

def parse_range(t):
    s = str(t).replace(",", "")
    m = re.search(r"(\d+\.?\d*)\s*-\s*(\d+\.?\d*)", s)
    if m: return float(m[1]), float(m[2])
    m = re.search(r"(\d+\.?\d*)", s)
    return (float(m[1]), float(m[1])) if m else (np.nan, np.nan)

VARS = {"temp": ("optimal_temperature_C", "temperature_status", "Temperature (°C)"),
        "rain": ("seasonal_rainfall_mm",  "rainfall_status",    "Seasonal rainfall (mm)"),
        "ph":   ("preferred_soil_pH",     "soil_pH_status",     "Soil pH"),
        "rh":   ("relative_humidity_percent", "humidity_status", "Relative humidity (%)")}
crops = df[["crop_name", "crop_group", "water_requirement"]].copy()
for k, (col, st, _) in VARS.items():
    r = df[col].apply(parse_range)
    crops[f"{k}_min"] = [x[0] for x in r]; crops[f"{k}_max"] = [x[1] for x in r]
    crops[f"{k}_doa"] = df[st].astype(str).str.startswith("DOA")
print(crops.shape, "| missing ranges:", int(crops[[f"{k}_min" for k in VARS]].isna().sum().sum()))

## 2. Download NASA POWER daily data
`T2M` air temperature, `RH2M` relative humidity, `PRECTOTCORR` bias-corrected precipitation (a POWER/MERRA-2-based product, not raw GPM IMERG). Grid cells are about 0.5°, so values describe the surrounding area, not the exact field.

In [ ]:
r = requests.get("https://power.larc.nasa.gov/api/temporal/daily/point", timeout=180, params=dict(
    parameters="T2M,RH2M,PRECTOTCORR", community="AG", latitude=LAT, longitude=LON,
    start=f"{START}0101", end=f"{END}1231", format="JSON"))
r.raise_for_status()
p = r.json()["properties"]["parameter"]
w = pd.DataFrame({"temp": p["T2M"], "rh": p["RH2M"], "rain": p["PRECTOTCORR"]})
w.index = pd.to_datetime(w.index, format="%Y%m%d")
w = w.replace(-999, np.nan).sort_index()
print(w.index.min().date(), "to", w.index.max().date(), "| missing days:", int(w.isna().any(axis=1).sum()))

## 3. Season summaries
Each season is summarised as mean temperature, total rainfall and mean humidity. Seasons that wrap the new year (Maha) are kept together; incomplete seasons are dropped.

In [ ]:
win = w[w.index.month.isin(SEASON_MONTHS)].copy()
wrap = SEASON_MONTHS[0] > SEASON_MONTHS[-1]
win["sy"] = (win.index.year - (win.index.month < SEASON_MONTHS[0]).astype(int)) if wrap else win.index.year
n = win.groupby("sy")["rain"].count()
full = n[n >= 0.95 * n.max()].index
S = win[win.sy.isin(full)].groupby("sy").agg(temp=("temp", "mean"), rh=("rh", "mean"), rain=("rain", "sum"))
S.index = S.index.astype(int)
print(len(S), "complete seasons:", S.index.min(), "to", S.index.max())
S.round(1).tail()

## 4. Are conditions really trending?
* **Theil–Sen slope** = robust trend per year.
* **Mann–Kendall p-value** (Kendall's tau vs. year): below 0.05 means a trend is statistically detectable. Above 0.05 means *no reliable trend*, so the extended line should not be trusted.
* Band = 80% prediction interval from a linear fit, extended to the target year. Interannual variability is large, so bands are wide.

In [ ]:
def trend(y, end_year):
    x = y.index.values.astype(float); v = y.values
    sen = stats.theilslopes(v, x); _, pval = stats.kendalltau(x, v)
    ols = sm.OLS(v, sm.add_constant(x)).fit()
    xf = np.arange(int(x.max()) + 1, end_year + 1).astype(float)
    xa = np.r_[x, xf]
    pf = ols.get_prediction(sm.add_constant(xa)).summary_frame(alpha=0.2)
    return dict(sen=sen[0], p=pval, x=xa, pf=pf, ols=ols, sd=np.sqrt(ols.scale), n_obs=len(x))

TR = {k: trend(S[k], TARGET_YEAR) for k in ["temp", "rain", "rh"]}
units = {"temp": "°C", "rain": "mm", "rh": "%"}
names = {"temp": "Mean temperature (°C)", "rain": "Seasonal rainfall (mm)", "rh": "Mean relative humidity (%)"}
cols  = {"temp": "#ff6b6b", "rain": "#4dabf7", "rh": "#69db7c"}

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07, subplot_titles=[names[k] for k in TR])
for i, k in enumerate(TR, 1):
    t = TR[k]; x = t["x"]; pf = t["pf"]
    fig.add_trace(go.Scatter(x=x, y=pf["obs_ci_upper"], line=dict(width=0), showlegend=False, hoverinfo="skip"), row=i, col=1)
    fig.add_trace(go.Scatter(x=x, y=pf["obs_ci_lower"], fill="tonexty", line=dict(width=0),
                             fillcolor="rgba(255,255,255,0.10)", name="80% band", showlegend=(i == 1), hoverinfo="skip"), row=i, col=1)
    fig.add_trace(go.Scatter(x=x, y=pf["mean"], mode="lines", line=dict(color=cols[k], width=3, dash="dot"),
                             name="Linear trend", showlegend=(i == 1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=S.index, y=S[k], mode="markers+lines", line=dict(color=cols[k], width=1),
                             marker=dict(size=7, color=cols[k], line=dict(color="white", width=1)),
                             name="Observed seasons", showlegend=(i == 1)), row=i, col=1)
    verdict = "significant trend" if t["p"] < 0.05 else "NO reliable trend"
    fig.add_annotation(xref=f"x{i if i>1 else ''} domain", yref=f"y{i if i>1 else ''} domain", x=0.01, y=0.97, showarrow=False,
                       xanchor="left", yanchor="top", align="left", font=dict(size=12, color="white"),
                       text=f"<b>{t['sen']*10:+.2f} {units[k]}/decade</b> | Mann-Kendall p = {t['p']:.3f} -> {verdict}",
                       bgcolor="rgba(0,0,0,0.5)")
    fig.add_vrect(x0=S.index.max() + 0.5, x1=TARGET_YEAR + 0.5, fillcolor="white", opacity=0.05, line_width=0, row=i, col=1)
fig.update_layout(template="plotly_dark", height=880, title=f"Climate trends and extension to {TARGET_YEAR}  |  {LAT}, {LON}  |  months {SEASON_MONTHS}",
                  legend=dict(orientation="h", y=1.07), margin=dict(t=110))
fig.show()

## 5. Rainfall rhythm: monthly anomalies (blue = wetter than usual, red = drier)
Each cell is one month's rainfall minus the long-term average for that calendar month, so droughts, wet spells and any drift over time are easy to spot.

In [ ]:
monthly = w["rain"].resample("MS").sum(min_count=25)
M = monthly.to_frame("rain"); M["year"] = M.index.year; M["month"] = M.index.month
clim = M.groupby("month")["rain"].mean()
M["anom"] = M["rain"] - M["month"].map(clim)
P = M.pivot(index="year", columns="month", values="anom")
mn = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
lim = float(np.nanpercentile(np.abs(P.values), 95))
hm = go.Figure(go.Heatmap(z=P.values, x=mn, y=P.index, colorscale="RdBu", zmid=0, zmin=-lim, zmax=lim,
                          colorbar=dict(title="mm vs avg"), xgap=1, ygap=1,
                          hovertemplate="%{y} %{x}: %{z:+.0f} mm vs average<extra></extra>"))
hm.update_yaxes(autorange="reversed")
hm.update_layout(template="plotly_dark", height=720, title="Monthly rainfall anomaly (NASA POWER PRECTOTCORR)")
hm.show()

## 6. How often did each crop's ranges actually match, and is that changing?
For each past season we check whether mean temperature, total rainfall and mean humidity each fell inside the crop's range. **"All three"** is the share of seasons where all were inside at once.
Dumbbell: earlier seasons (grey) vs. the most recent seasons (colored). Seasonal *means* can hide heat spells and dry spells, so this is a screening measure. Soil pH is excluded because it is not climate-driven.

In [ ]:
def inrange(series, lo, hi): return (series >= lo) & (series <= hi)
rows = []
for _, c in crops.iterrows():
    ok = {k: inrange(S[k], c[f"{k}_min"], c[f"{k}_max"]) for k in ["temp", "rain", "rh"]}
    allok = ok["temp"] & ok["rain"] & ok["rh"]
    rows.append(dict(crop=c.crop_name, group=c.crop_group,
                     all_freq=allok.mean(), early=allok.iloc[:-RECENT_N].mean(), recent=allok.iloc[-RECENT_N:].mean(),
                     **{f"{k}_freq": ok[k].mean() for k in ok}))
H = pd.DataFrame(rows).sort_values(["recent", "all_freq"]).reset_index(drop=True)
H["change"] = H["recent"] - H["early"]

fig = go.Figure()
xs, ys = [], []
for _, r_ in H.iterrows(): xs += [r_.early, r_.recent, None]; ys += [r_.crop, r_.crop, None]
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color="rgba(255,255,255,0.35)", width=3), hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=H.early, y=H.crop, mode="markers", name=f"Earlier seasons", marker=dict(size=10, color="#868e96")))
fig.add_trace(go.Scatter(x=H.recent, y=H.crop, mode="markers", name=f"Last {RECENT_N} seasons",
                         marker=dict(size=13, color=H.change, colorscale="RdYlGn", cmin=-0.5, cmax=0.5, line=dict(color="white", width=1),
                                     colorbar=dict(title="Change")),
                         hovertemplate="%{y}<br>recent: %{x:.0%}<extra></extra>"))
fig.update_xaxes(tickformat=".0%", range=[-0.03, 1.03], title="Share of seasons with temperature, rainfall and humidity all in range")
fig.update_layout(template="plotly_dark", height=max(600, 22 * len(H)), title="Historical match frequency by crop (earlier vs. recent)", margin=dict(l=170))
fig.show()
H[["crop", "all_freq", "early", "recent", "change"]].sort_values("recent", ascending=False).head(10).round(2)

## 7. Projected chance each variable is in range in the target year
Uses the linear trend plus residual variability (normal approximation). Variables are shown separately because they are not independent. If a variable had no significant trend in step 4, treat its projection as roughly "the same as history".

In [ ]:
def proj_prob(k, lo, hi):
    t = TR[k]; X = sm.add_constant(np.array([float(TARGET_YEAR)]), has_constant="add")
    pr = t["ols"].get_prediction(X).summary_frame()
    mu = float(pr["mean"].iloc[0]); sd = float(np.sqrt(t["sd"]**2 + pr["mean_se"].iloc[0]**2))
    return stats.norm.cdf(hi, mu, sd) - stats.norm.cdf(lo, mu, sd)

PR = pd.DataFrame({k: [proj_prob(k, c[f"{k}_min"], c[f"{k}_max"]) for _, c in crops.iterrows()] for k in ["temp", "rain", "rh"]},
                  index=crops.crop_name)
PR["avg"] = PR.mean(axis=1); PR = PR.sort_values("avg")
fig = go.Figure(go.Heatmap(z=PR[["temp", "rain", "rh"]].values, x=["Temperature", "Rainfall", "Humidity"], y=PR.index,
                           text=np.round(PR[["temp", "rain", "rh"]].values * 100).astype(int), texttemplate="%{text}%",
                           colorscale="Viridis", zmin=0, zmax=1, xgap=2, ygap=2, colorbar=dict(title="P(in range)", tickformat=".0%")))
fig.update_layout(template="plotly_dark", height=max(600, 22 * len(PR)), title=f"Projected chance of being in range in {TARGET_YEAR}", margin=dict(l=170))
fig.show()

## 8. Machine learning A: map the crops (PCA + K-means), and place your farm on the map
Unsupervised learning groups crops with similar requirement profiles. Features are the midpoints of temperature, rainfall, pH and humidity (standardised), so your farm can be placed in the same space using its average season conditions and your soil-test pH.

This shows *which crop profiles resemble your site's conditions*. It is **not** a yield prediction, and range widths are ignored here, so cross-check with Step 6.

In [ ]:
F = pd.DataFrame({k: (crops[f"{k}_min"] + crops[f"{k}_max"]) / 2 for k in ["temp", "rain", "ph", "rh"]})
farm = pd.DataFrame([{"temp": S["temp"].mean(), "rain": S["rain"].mean(), "ph": np.mean(PH_BAND), "rh": S["rh"].mean()}])
sc = StandardScaler().fit(F)
Z = sc.transform(F); zf = sc.transform(farm)

best_k, best_s = 2, -1
for k in range(2, 7):
    lab = KMeans(k, n_init=20, random_state=0).fit_predict(Z); s_ = silhouette_score(Z, lab)
    if s_ > best_s: best_k, best_s = k, s_
km = KMeans(best_k, n_init=20, random_state=0).fit(Z); crops["cluster"] = km.labels_
print(f"K-means clusters: {best_k} (silhouette {best_s:.2f}; above ~0.5 is clear structure, below ~0.25 is weak)")

pca = PCA(2).fit(Z); XY = pca.transform(Z); xyf = pca.transform(zf)
print("variance explained by 2 components:", np.round(pca.explained_variance_ratio_, 2))
load = pd.DataFrame(pca.components_.T, index=F.columns, columns=["PC1", "PC2"]).round(2); print(load)

dist = np.linalg.norm(Z - zf, axis=1); near = np.argsort(dist)[:5]
fig = go.Figure()
pal = ["#4dabf7", "#ff922b", "#69db7c", "#da77f2", "#ffd43b", "#ff6b6b"]
for cl in range(best_k):
    m = (crops.cluster == cl).values
    fig.add_trace(go.Scatter(x=XY[m, 0], y=XY[m, 1], mode="markers+text", text=crops.crop_name[m], textposition="top center",
                             textfont=dict(size=9), name=f"Cluster {cl + 1}", marker=dict(size=12, color=pal[cl], line=dict(color="white", width=1)),
                             customdata=np.c_[F.temp[m], F.rain[m], F.ph[m], F.rh[m]],
                             hovertemplate="%{text}<br>T %{customdata[0]:.0f}°C, rain %{customdata[1]:.0f} mm, pH %{customdata[2]:.1f}, RH %{customdata[3]:.0f}%<extra></extra>"))
for j in near:
    fig.add_trace(go.Scatter(x=[xyf[0, 0], XY[j, 0]], y=[xyf[0, 1], XY[j, 1]], mode="lines",
                             line=dict(color="gold", width=1.5, dash="dot"), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=xyf[:, 0], y=xyf[:, 1], mode="markers+text", text=["YOUR FARM"], textposition="bottom center",
                         marker=dict(size=24, symbol="star", color="gold", line=dict(color="white", width=2)), name="Your farm"))
fig.update_layout(template="plotly_dark", height=720, title="Crop requirement map (PCA) with your farm and its 5 nearest crop profiles",
                  xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]:.0%})", yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]:.0%})")
fig.show()
pd.DataFrame({"crop": crops.crop_name.iloc[near].values, "distance": dist[near].round(2), "cluster": crops.cluster.iloc[near].values + 1})

## 9. Machine learning B: back-tested rainfall forecast
Monthly rainfall is noisy. Before trusting any forecast we **test it on data it has not seen** (the last 24 months) and compare it with a simple baseline (the long-term average for each calendar month). A model that does not beat the baseline is not adding information.

In [ ]:
ms = monthly.dropna().asfreq("MS")
train, test = ms.iloc[:-24], ms.iloc[-24:]

clim_tr = train.groupby(train.index.month).mean()
base = pd.Series(test.index.month.map(clim_tr).values, index=test.index)
hw = ExponentialSmoothing(train, trend=None, seasonal="add", seasonal_periods=12, initialization_method="estimated").fit()
pred = hw.forecast(24).clip(lower=0)

mae = lambda a, b: float(np.mean(np.abs(a - b)))
mae_base, mae_hw = mae(test, base), mae(test, pred)
print(f"Back-test MAE (mm/month): climatology baseline {mae_base:.1f} | Holt-Winters {mae_hw:.1f}")
use_hw = mae_hw < mae_base
print("Holt-Winters beats the baseline -> forecast uses it." if use_hw else "Holt-Winters does NOT beat the baseline -> forecast uses the climatology baseline (honest result).")

# refit on all data, forecast 12 months
clim_all = ms.groupby(ms.index.month).mean()
idx = pd.date_range(ms.index[-1] + pd.offsets.MonthBegin(1), periods=12, freq="MS")
if use_hw:
    fc = ExponentialSmoothing(ms, trend=None, seasonal="add", seasonal_periods=12, initialization_method="estimated").fit().forecast(12).clip(lower=0)
else:
    fc = pd.Series(idx.month.map(clim_all).values, index=idx)
sd = float(np.std(test - (pred if use_hw else base)))
fig = go.Figure()
fig.add_trace(go.Scatter(x=ms.index[-84:], y=ms.iloc[-84:], mode="lines", name="Observed", line=dict(color="#4dabf7", width=2)))
fig.add_trace(go.Scatter(x=test.index, y=(pred if use_hw else base), mode="lines", name="Back-test (unseen months)", line=dict(color="#ffd43b", dash="dash")))
fig.add_trace(go.Scatter(x=fc.index, y=(fc + 1.28 * sd).clip(lower=0), line=dict(width=0), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=fc.index, y=(fc - 1.28 * sd).clip(lower=0), fill="tonexty", line=dict(width=0), fillcolor="rgba(255,107,107,0.2)", name="~80% band"))
fig.add_trace(go.Scatter(x=fc.index, y=fc, mode="lines+markers", name="12-month forecast", line=dict(color="#ff6b6b", width=3)))
fig.update_layout(template="plotly_dark", height=480, title=f"Monthly rainfall forecast (back-test MAE: baseline {mae_base:.0f} mm vs model {mae_hw:.0f} mm)", yaxis_title="mm / month")
fig.show()

## How to read the results responsibly
* **Step 4 verdict** decides whether a trend extension means anything. "NO reliable trend" means the extension is just noise.
* **Step 6** is based on history only; **Step 7** extends trends. Compare them.
* **Step 8** compares requirement *profiles*. Crop ranges are mostly estimates (see the DOA counts in the earlier notebook), so clusters describe the table, not proven agronomy.
* NASA POWER is gridded reanalysis/satellite-based data. Validate against local rain-gauge and farm records.
* Nothing here replaces advice from local agricultural officers and farmers.